# Selected-Week Model Comparison

Scope:
- hourly
- D-only
- DA-only stochastic bidding
- risk-neutral only

This notebook reads a completed selected-week suite output folder. It does **not** rerun the MILPs.

Selected weeks are diagnostic support statements, not an unbiased full-period evaluation.


## Artifacts

Main comparison target:
1. LEAR Strict
2. LEAR FS3 pruned candidate
3. XGBoost FS3 pruned candidate

If the loaded suite contains fewer models, that reflects an upstream support or provenance blocker rather than notebook filtering.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

repo_root = Path.cwd()
while repo_root.name != 'Thesis' and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from visual_style import MODEL_COLORS, STRATEGY_COLORS, apply_visual_style
    apply_visual_style()
except Exception:
    MODEL_COLORS = {
        'LEAR Strict': '#C97941',
        'LEAR FS3 pruned candidate': '#3A7D7C',
        'XGBoost FS3 pruned candidate': '#1F4E79',
        'Price insensitive benchmark': '#333333',
    }
    STRATEGY_COLORS = {'price_insensitive': '#333333'}

MODEL_ORDER = ['LEAR Strict', 'LEAR FS3 pruned candidate', 'XGBoost FS3 pruned candidate']
WEEK_ORDER = ['high_volatility_week', 'stable_summer_week', 'stable_winter_week', 'low_price_week', 'tail_week']

SUITE_DIR = Path(r'C:/Users/marijnvalk/PycharmProjects/Thesis/scripts/Data/03_Hydrogen_Test_Case/runs/20260516_122310_hydrogen_phase6d_selected_week_real_scenarios')
NOTEBOOK_INPUTS = SUITE_DIR / 'notebook_inputs'
selected_week_registry = pd.read_csv(SUITE_DIR / 'selected_week_registry.csv')
daily_metrics = pd.read_csv(SUITE_DIR / 'daily_metrics.csv')
weekly_metrics = pd.read_csv(SUITE_DIR / 'weekly_metrics.csv')
weekly_metrics_by_model = pd.read_csv(SUITE_DIR / 'weekly_metrics_by_model.csv')
benchmark_comparison_daily = pd.read_csv(SUITE_DIR / 'benchmark_comparison_daily.csv')
benchmark_comparison_weekly = pd.read_csv(SUITE_DIR / 'benchmark_comparison_weekly.csv')
validation_checks = pd.read_csv(SUITE_DIR / 'validation_checks_all_runs.csv')
model_stats_daily = pd.read_csv(SUITE_DIR / 'model_stats_daily.csv')
scenario_fan_inputs = pd.read_parquet(NOTEBOOK_INPUTS / 'scenario_fan_inputs.parquet') if (NOTEBOOK_INPUTS / 'scenario_fan_inputs.parquet').exists() else pd.DataFrame()
redispatch_timeseries = pd.read_parquet(NOTEBOOK_INPUTS / 'actual_redispatch_timeseries_daily.parquet') if (NOTEBOOK_INPUTS / 'actual_redispatch_timeseries_daily.parquet').exists() else pd.DataFrame()


## Method Overview

Pipeline:
`scenario set -> bid optimisation -> actual clearing -> deterministic redispatch -> realised settlement`

Important market-chain rule:
- bid price affects acceptance;
- accepted energy is settled at the realised DA market price;
- redispatch can only use cleared electricity;
- unused cleared energy and shortfall remain explicit in the realised metrics.

The benchmark is `price_insensitive_plan_first_market_cap`, which uses the same realised delivery day and actual price path.


## Selected Week Registry


In [ ]:
selected_week_registry


## Metric Guide

- `realised_adjusted_profit`: realised hydrogen revenue minus actual DA settlement cost, unused-energy penalty, shortfall penalty, and terminal inventory correction.
- `stochastic_minus_benchmark_profit`: realised stochastic profit minus realised price-insensitive benchmark profit.
- `clearing_ratio`: cleared energy divided by submitted bid energy.
- `rejected_energy_mwh`: submitted energy that did not clear.
- `weighted_average_actual_price_paid`: realised DA settlement cost divided by cleared energy.
- `hydrogen_compressed_or_sold_kg`: realised hydrogen delivered out of redispatch.
- `hydrogen_above_target_kg`: realised hydrogen above the lower-bound daily target.
- `target_fulfilment_ratio_capped_for_reliability`: reliability-style fulfilment capped at 1.0.
- `shortfall_kg`: unmet daily hydrogen target after redispatch.
- `unused_cleared_energy_mwh`: cleared DA electricity that could not be physically used.
- `terminal_inventory_correction_eur`: end-of-window inventory adjustment.
- `worst_scenario_profit`: worst scenario-side adjusted profit from the bid optimisation step.
- `expected_vs_realised_profit_delta`: realised adjusted profit minus the expected stochastic objective.


## Overview Results


In [ ]:
overview_cols = [
    'week_label', 'model_label', 'realised_adjusted_profit',
    'price_insensitive_realised_adjusted_profit', 'stochastic_minus_benchmark_profit',
    'hydrogen_compressed_or_sold_kg', 'shortfall_kg', 'clearing_ratio',
    'weighted_average_actual_price_paid'
]
overview = weekly_metrics[overview_cols].copy()
overview = overview.sort_values([
    overview['week_label'].map({label: idx for idx, label in enumerate(WEEK_ORDER)}),
    overview['model_label'].map({label: idx for idx, label in enumerate(MODEL_ORDER)})
])
display(overview.round(2))


In [ ]:
def plot_week_scenario_fan(week_label: str):
    if scenario_fan_inputs.empty:
        return
    week_frame = scenario_fan_inputs[scenario_fan_inputs['week_label'] == week_label].copy()
    models = [label for label in MODEL_ORDER if label in week_frame['model_label'].astype(str).unique().tolist()]
    fig, axes = plt.subplots(len(models), 1, figsize=(12, max(3.6, 3.4 * len(models))), sharex=True)
    if len(models) == 1:
        axes = [axes]
    for ax, model_label in zip(axes, models):
        model_frame = week_frame[week_frame['model_label'] == model_label].copy()
        rows = []
        for ts, grp in model_frame.groupby('delivery_start_utc', sort=True):
            vals = grp['scenario_price_eur_per_mwh'].astype(float).to_numpy()
            probs = grp['scenario_probability'].astype(float).to_numpy()
            order = np.argsort(vals)
            vals = vals[order]
            probs = probs[order]
            cum = np.cumsum(probs)
            def q(alpha: float) -> float:
                idx = min(np.searchsorted(cum, alpha, side='left'), len(vals) - 1)
                return float(vals[idx])
            rows.append({
                'delivery_start_utc': pd.Timestamp(ts),
                'q05': q(0.05),
                'q50': q(0.50),
                'q95': q(0.95),
                'actual': float(grp['actual_price_eur_per_mwh'].iloc[0]),
            })
        quant = pd.DataFrame(rows).sort_values('delivery_start_utc')
        color = MODEL_COLORS.get(model_label, '#7A7A7A')
        ax.fill_between(quant['delivery_start_utc'], quant['q05'], quant['q95'], alpha=0.22, color=color)
        ax.plot(quant['delivery_start_utc'], quant['q50'], color=color, linewidth=1.6, label=f'{model_label} p50')
        ax.plot(quant['delivery_start_utc'], quant['actual'], color='#222222', linewidth=1.8, label='Actual price')
        ax.set_ylabel('EUR/MWh')
        ax.set_title(f'{week_label}: {model_label}')
        ax.legend(loc='upper left')
    axes[-1].set_xlabel('Delivery hour')
    fig.tight_layout()
    plt.show()

def grouped_week_bar(metric: str, ylabel: str, title: str):
    frame = weekly_metrics.copy()
    weeks = [label for label in WEEK_ORDER if label in frame['week_label'].astype(str).unique().tolist()]
    x = np.arange(len(weeks))
    width = 0.22
    fig, ax = plt.subplots(figsize=(12, 5.2))
    present_models = [label for label in MODEL_ORDER if label in frame['model_label'].astype(str).unique().tolist()]
    for idx, model_label in enumerate(present_models):
        subset = frame[frame['model_label'] == model_label].set_index('week_label').reindex(weeks)
        ax.bar(x + (idx - (len(present_models)-1)/2) * width, subset[metric].to_numpy(), width=width, color=MODEL_COLORS.get(model_label, '#7A7A7A'), label=model_label)
    ax.set_xticks(x)
    ax.set_xticklabels(weeks, rotation=15)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    plt.show()


## Week-by-Week Visual Diagnostics


In [ ]:
for week_label in selected_week_registry['week_label'].tolist():
    display(Markdown(f'### {week_label}'))
    week_table = weekly_metrics[weekly_metrics['week_label'] == week_label][[
        'model_label', 'realised_adjusted_profit', 'price_insensitive_realised_adjusted_profit',
        'stochastic_minus_benchmark_profit', 'clearing_ratio', 'rejected_energy_mwh',
        'hydrogen_compressed_or_sold_kg', 'shortfall_kg', 'weighted_average_actual_price_paid'
    ]].copy()
    display(week_table.round(2))
    display(Markdown('Support statement: compare realised DA prices and scenario spread first, then check whether differences come from clearing, price paid, hydrogen delivery, or shortfall.'))
    plot_week_scenario_fan(week_label)
    grouped_week_bar('realised_adjusted_profit', 'EUR', f'Realised adjusted profit in {week_label}')
    grouped_week_bar('clearing_ratio', 'Ratio', f'Clearing ratio in {week_label}')
    grouped_week_bar('rejected_energy_mwh', 'MWh', f'Rejected energy in {week_label}')
    grouped_week_bar('hydrogen_compressed_or_sold_kg', 'kg', f'Hydrogen compressed/sold in {week_label}')
    grouped_week_bar('shortfall_kg', 'kg', f'Shortfall in {week_label}')
    grouped_week_bar('weighted_average_actual_price_paid', 'EUR/MWh', f'Average actual price paid in {week_label}')


## Cross-Week Model Comparison


In [ ]:
pivot = weekly_metrics.pivot(index='week_label', columns='model_label', values='stochastic_minus_benchmark_profit').reindex(WEEK_ORDER)
pivot = pivot[[col for col in MODEL_ORDER if col in pivot.columns]]
fig, ax = plt.subplots(figsize=(9, 4.8))
vals = pivot.to_numpy(dtype=float)
image = ax.imshow(vals, aspect='auto', cmap='RdYlGn')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(list(pivot.columns), rotation=20, ha='right')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(list(pivot.index))
for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        if np.isfinite(vals[i, j]):
            ax.text(j, i, f'{vals[i, j]:.0f}', ha='center', va='center', fontsize=8)
fig.colorbar(image, ax=ax, label='Stochastic - benchmark profit (EUR)')
ax.set_title('Cross-week stochastic uplift heatmap')
fig.tight_layout()
plt.show()

grouped_week_bar('realised_adjusted_profit', 'EUR', 'Realised adjusted profit by week and model')

scatter = weekly_metrics[['model_label', 'actual_price_spread', 'stochastic_minus_benchmark_profit']].copy()
fig, ax = plt.subplots(figsize=(8.5, 5.0))
for model_label, grp in scatter.groupby('model_label'):
    ax.scatter(grp['actual_price_spread'], grp['stochastic_minus_benchmark_profit'], label=model_label, color=MODEL_COLORS.get(model_label, '#7A7A7A'), s=60)
ax.axhline(0.0, color='#333333', linewidth=1.0)
ax.set_xlabel('Weekly actual price spread (EUR/MWh)')
ax.set_ylabel('Stochastic - benchmark profit (EUR)')
ax.set_title('Does wider realised spread coincide with stochastic uplift?')
ax.legend()
fig.tight_layout()
plt.show()

display(model_stats_daily.groupby('model_label')[['solve_time_seconds', 'variables', 'binaries', 'constraints', 'scenario_count']].agg(['mean', 'max']).round(2))


## Interpretation

Use the figures in this order:
1. compare actual prices and each model’s scenario fan;
2. compare realised profit versus the price-insensitive benchmark;
3. check whether the difference came from clearing ratio, rejected energy, price paid, hydrogen delivery, or shortfall;
4. keep the selected-week caveat explicit.

Selected weeks can show mechanism and failure modes. They are not a claim about representative average performance over the full evaluation period.


## Validation and Limitations

- hard validation checks should be read from `validation_checks_all_runs.csv`;
- submitted bids are non-anticipative and contain no `scenario_id`;
- clearing is pay-as-cleared against realised DA prices;
- redispatch respects `used + unused = cleared`;
- no mFRR, no quarter-hour, and no CVaR were included in this selected-week suite;
- if LEAR Strict is missing from the loaded suite, that indicates a common-support blocker in the artifact coverage, not a notebook omission.


In [ ]:
validation_summary = validation_checks.groupby(['status']).size().rename('count').reset_index()
display(validation_summary)
display(validation_checks.head())
